# Class-Conditional Flow Matching vs Guided Flow Matching (MC Guidance)

This notebook demonstrates the comparison between:
1. **Explicit Class-Conditional Flow Matching (CCFM)**: The vector field model $v(x_t, t, y)$ is trained with explicit conditioning on class labels $y \in \{0, 1, ..., 7\}$.
2. **Guided Flow Matching (GFM) via Monte Carlo Sampling**: The vector field model $v(x_t, t)$ is trained unconditionally. At inference time, we guide the flow towards a specific class $c$ using Monte Carlo importance sampling over predicted endpoints.

Both methods are evaluated on the 2D **8-Gaussians** mixture distribution.

In [ ]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from torchdyn.core import NeuralODE

from guided_flow.backbone.mlp import MLP, ClassConditionalMLP
from guided_flow.backbone.wrapper import GuidedMLPWrapper, MLPWrapper, ClassConditionalMLPWrapper
from guided_flow.distributions.base import get_distribution
from guided_flow.utils.misc import deterministic, get_device
from guided_flow.utils.metrics import compute_w2

device = get_device()
print("Using device:", device)
deterministic(42)

## 1. Data Generation and Visualization
We sample from the 8-Gaussians target distribution, where each Gaussian represents a different class mode.

In [ ]:
# Initialize distributions
x0_dist = get_distribution('gaussian')
x1_dist = get_distribution('8gaussian')

# Sample points
x0_samples = x0_dist.sample(2000, device=device)
x1_samples = x1_dist.sample(2000, device=device)
y_samples = x1_dist.get_J(x1_samples).long().cpu().numpy()

# Plot the distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(x0_samples[:, 0].cpu().numpy(), x0_samples[:, 1].cpu().numpy(), alpha=0.5, color='royalblue', edgecolors='k')
axes[0].set_title('Source Distribution: Standard Gaussian ($p_0$)', fontsize=14)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)
axes[0].set_aspect('equal')
axes[0].grid(True, linestyle='--', alpha=0.5)

scatter = axes[1].scatter(x1_samples[:, 0].cpu().numpy(), x1_samples[:, 1].cpu().numpy(), c=y_samples, cmap='tab10', alpha=0.7, edgecolors='k')
axes[1].set_title('Target Distribution: 8-Gaussians ($p_1$)', fontsize=14)
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect('equal')
axes[1].grid(True, linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Center Mode Index ($y$)', fontsize=12)

plt.show()

## 2. Model Training with Loss Tracking
We define custom training loops for both the **Unconditional CFM** and **Class-Conditional Flow Matching (CCFM)** models, allowing us to record and plot their loss curves during optimization.

In [ ]:
from guided_flow.flow.conditional_flow_matching import get_cfm

def train_unconditional_with_losses(
    x0_sampler, x1_sampler, cfm='cfm', width=256, sigma=0.0, device=None, batch_size=1024, lr=1e-3, num_steps=15000
):
    model = MLP(dim=2, w=width, time_varying=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    FM = get_cfm(cfm, sigma)
    
    losses = []
    with tqdm(range(num_steps), desc="Training Unconditional CFM") as pbar:
        for k in pbar:
            optimizer.zero_grad()
            x0 = x0_sampler(batch_size=batch_size, device=device)
            x1 = x1_sampler(batch_size=batch_size, device=device)
            
            t = torch.rand(x0.shape[0]).to(device)
            t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1, t)
            
            vt = model(torch.cat([xt, t[:, None]], dim=-1))
            loss = torch.mean((vt - ut) ** 2)
            
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            
            if k % 100 == 0:
                pbar.set_postfix(loss=loss.item())
    return model, losses

def train_conditional_with_losses(
    x0_sampler, x1_sampler, x1_dist, cfm='cfm', width=256, sigma=0.0, device=None, batch_size=1024, lr=1e-3, num_steps=15000
):
    model = ClassConditionalMLP(dim=2, num_classes=8, w=width, time_varying=True, class_emb_dim=16).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    FM = get_cfm(cfm, sigma)
    
    losses = []
    with tqdm(range(num_steps), desc="Training Conditional CCFM") as pbar:
        for k in pbar:
            optimizer.zero_grad()
            x0 = x0_sampler(batch_size=batch_size, device=device)
            x1 = x1_sampler(batch_size=batch_size, device=device)
            
            t = torch.rand(x0.shape[0]).to(device)
            t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1, t)
            
            # Reconstruct coupled x1 to ensure label alignment
            x1_coupled = xt + (1.0 - t[:, None]) * ut
            y = x1_dist.get_J(x1_coupled).long().to(device)
            
            vt = model(torch.cat([xt, t[:, None]], dim=-1), y)
            loss = torch.mean((vt - ut) ** 2)
            
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            
            if k % 100 == 0:
                pbar.set_postfix(loss=loss.item())
    return model, losses

In [ ]:
FORCE_RETRAIN = True
NUM_STEPS = 15000
BATCH_SIZE = 1024
WIDTH = 256
LR = 1e-3

log_dir = '../logs/gaussian-8gaussian/cfm_gaussian_8gaussian'
os.makedirs(log_dir, exist_ok=True)

uncond_model_path = os.path.join(log_dir, 'cfm_gaussian_8gaussian.pth')
cond_model_path = os.path.join(log_dir, 'ccfm_gaussian_8gaussian.pth')

uncond_losses = None
cond_losses = None

# 1. Unconditional model training/loading
uncond_model = MLP(dim=2, w=WIDTH, time_varying=True).to(device)
if os.path.exists(uncond_model_path) and not FORCE_RETRAIN:
    print("Loading pre-trained unconditional model...")
    uncond_model.load_state_dict(torch.load(uncond_model_path, map_location=device))
else:
    print("Training unconditional model...")
    uncond_model, uncond_losses = train_unconditional_with_losses(
        x0_dist.sample, x1_dist.sample, cfm='cfm', width=WIDTH, device=device, batch_size=BATCH_SIZE, lr=LR, num_steps=NUM_STEPS
    )
    torch.save(uncond_model.state_dict(), uncond_model_path)
    print("Unconditional model saved.")

# 2. Class-Conditional model training/loading
cond_model = ClassConditionalMLP(dim=2, num_classes=8, w=WIDTH, time_varying=True, class_emb_dim=16).to(device)
if os.path.exists(cond_model_path) and not FORCE_RETRAIN:
    print("Loading pre-trained class-conditional model...")
    cond_model.load_state_dict(torch.load(cond_model_path, map_location=device))
else:
    print("Training class-conditional model...")
    cond_model, cond_losses = train_conditional_with_losses(
        x0_dist.sample, x1_dist.sample, x1_dist, cfm='cfm', width=WIDTH, device=device, batch_size=BATCH_SIZE, lr=LR, num_steps=NUM_STEPS
    )
    torch.save(cond_model.state_dict(), cond_model_path)
    print("Class-conditional model saved.")

### Loss Curve Visualization
We plot the MSE training losses below if the models were retrained.

In [ ]:
if uncond_losses is not None or cond_losses is not None:
    plt.figure(figsize=(12, 6))
    if uncond_losses is not None:
        # Apply window smoothing for visualization clarity
        uncond_smooth = np.convolve(uncond_losses, np.ones(100)/100, mode='valid')
        plt.plot(uncond_smooth, label='Unconditional CFM Loss (smoothed)', alpha=0.8, color='royalblue')
    if cond_losses is not None:
        cond_smooth = np.convolve(cond_losses, np.ones(100)/100, mode='valid')
        plt.plot(cond_smooth, label='Class-Conditional FM Loss (smoothed)', alpha=0.8, color='darkorange')
    plt.title('Training Loss Curves Comparison', fontsize=14)
    plt.xlabel('Gradient Steps', fontsize=12)
    plt.ylabel('Mean Squared Error', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()
else:
    print("Loaded pre-trained weights. No active training occurred to record losses.")

## 3. Class-Conditional Flow Matching (CCFM) Sampling
We sample from the trained CCFM model for a specific target class. We pass the class label explicitly at each step of the ODE solver.

In [ ]:
def sample_ccfm(model, class_label, batch_size=1000, num_steps=100):
    y = torch.full((batch_size,), class_label, dtype=torch.long, device=device)
    wrapper = ClassConditionalMLPWrapper(model, y)
    node = NeuralODE(wrapper, solver='euler')
    
    x0 = x0_dist.sample(batch_size, device=device)
    t_span = torch.linspace(0, 1, num_steps)
    with torch.no_grad():
        traj = node.trajectory(x0, t_span=t_span)
    return traj

## 4. Guided Flow Matching (Monte Carlo Guidance)
At time $t$, we predict the final target destination $\hat{x}_1 = x_t + v(x_t, t)(1-t)$. We then sample $M$ candidate endpoints $x_1^{(j)} \sim \mathcal{N}(\hat{x}_1, \sigma^2 I)$ and use Monte Carlo importance sampling to guide the trajectory via the energy function $J(x_1)$.

The guided vector field is:
$$v_{\text{guided}}(x_t, t) = v(x_t, t) + s \cdot g_t(x_t)$$
where $g_t(x_t) = \mathbb{E}_{j} \left[ \left( \frac{e^{-s J(x_1^{(j)})}}{Z} - 1 \right) \frac{x_1^{(j)} - x_t}{1-t+\epsilon} \right]$.

We compare two choices for the target energy $J_c(x_1)$ to guide the model towards class $c$:
1. **Indicator Guidance**: $J_c(x_1) = \mathbb{I}(\text{class}(x_1) \neq c)$. The importance weights are 1 if a candidate sample is classified as class $c$, and $e^{-scale}$ otherwise.
2. **Distance Guidance**: $J_c(x_1) = \|x_1 - \text{center}_c\|^2$. The candidate samples are weighted by their distance to the center of mode $c$.

In [ ]:
from guided_flow.guidance.gradient_guidance import wrap_grad_fn

def get_class_mc_guide_fn(x1_dist, target_class, scale, guide_scale=None, mc_n=100, std=1.5, ep=1e-2, schedule='linear_decay'):
    if guide_scale is None:
        guide_scale = scale

    def J_class_indicator(x1):
        classes = x1_dist.get_J(x1)
        return (classes != target_class).float()

    def guide_fn(t, x, dx_dt, model):
        x1_pred = x + dx_dt * (1 - t)
        
        # Bug Fix 1: Shrink candidate perturbation variance as t -> 1 to avoid exploding velocity variance
        perturb_std = std * (1.0 - t)
        x1_samples = torch.randn(mc_n, x.shape[0], x.shape[1], device=x.device) * perturb_std + x1_pred.unsqueeze(0)
        
        energies = J_class_indicator(x1_samples.reshape(-1, 2)).reshape(mc_n, -1)
        
        # Bug Fix 2: Shift energies by min to prevent numerical underflow / weight freezing
        energies_shifted = energies - energies.min(dim=0, keepdim=True)[0]
        weights = torch.exp(-scale * energies_shifted)
        
        v = (x1_samples - x) / (1 - t + ep)
        Z = weights.mean(0) + 1e-8
        g = ((weights / Z.unsqueeze(0)) - 1).unsqueeze(2) * v
        return g.mean(0)
        
    # Bug Fix 4: Pass guide_scale here for external magnitude scaling in wrap_grad_fn
    return wrap_grad_fn(guide_scale, schedule, guide_fn)

def get_distance_mc_guide_fn(x1_dist, target_class, scale, guide_scale=None, mc_n=100, std=0.1, ep=1e-2, schedule='linear_decay'):
    if guide_scale is None:
        guide_scale = scale
    center = x1_dist.centers[target_class].to(device)

    def guide_fn(t, x, dx_dt, model):
        x1_pred = x + dx_dt * (1 - t)
        
        # Bug Fix 1: Shrink candidate perturbation variance as t -> 1 to avoid exploding velocity variance
        perturb_std = std * (1.0 - t)
        x1_samples = torch.randn(mc_n, x.shape[0], x.shape[1], device=x.device) * perturb_std + x1_pred.unsqueeze(0)
        
        # Distance energy
        energies = torch.sum((x1_samples - center) ** 2, dim=-1)
        
        # Bug Fix 2: Shift energies by min to prevent numerical underflow / weight freezing
        energies_shifted = energies - energies.min(dim=0, keepdim=True)[0]
        weights = torch.exp(-scale * energies_shifted)
        
        v = (x1_samples - x) / (1 - t + ep)
        Z = weights.mean(0) + 1e-8
        g = ((weights / Z.unsqueeze(0)) - 1).unsqueeze(2) * v
        return g.mean(0)
        
    # Bug Fix 4: Pass guide_scale here for external magnitude scaling in wrap_grad_fn
    return wrap_grad_fn(guide_scale, schedule, guide_fn)

def sample_guided_gfm(uncond_model, target_class, guide_type='indicator', scale=5.0, guide_scale=None, batch_size=1000, num_steps=100, mc_n=100, std=0.1):
    if guide_scale is None:
        guide_scale = scale
        
    if guide_type == 'indicator':
        guide_fn = get_class_mc_guide_fn(x1_dist, target_class, scale, guide_scale=guide_scale, mc_n=mc_n, std=std)
    elif guide_type == 'distance':
        guide_fn = get_distance_mc_guide_fn(x1_dist, target_class, scale, guide_scale=guide_scale, mc_n=mc_n, std=std)
    else:
        raise ValueError("Unknown guide_type")
        
    wrapper = GuidedMLPWrapper(uncond_model, guide_fn=guide_fn)
    node = NeuralODE(wrapper, solver='euler')
    
    x0 = x0_dist.sample(batch_size, device=device)
    t_span = torch.linspace(0, 1, num_steps, device=device)
    with torch.no_grad():
        traj = node.trajectory(x0, t_span=t_span)
    return traj

## 5. Comparative Evaluation
We sample from all 8 target classes using CCFM, GFM (Indicator Guidance) and GFM (Distance Guidance).

In [ ]:
BATCH_SIZE_EVAL = 500
NUM_STEPS_EVAL = 100

ccfm_trajs = {}
gfm_indicator_trajs = {}
gfm_distance_trajs = {}

print("Sampling from all 8 classes...")
for c in tqdm(range(8)):
    # 1. Class-Conditional Flow Matching
    ccfm_trajs[c] = sample_ccfm(cond_model, class_label=c, batch_size=BATCH_SIZE_EVAL, num_steps=NUM_STEPS_EVAL)
    
    # 2. GFM via Indicator MC Guidance (Scale = 12.0 for sharp selection, guide_scale = 12.0)
    gfm_indicator_trajs[c] = sample_guided_gfm(
        uncond_model, 
        target_class=c, 
        guide_type='indicator', 
        scale=12.0, 
        guide_scale=15.0,
        batch_size=BATCH_SIZE_EVAL, 
        num_steps=NUM_STEPS_EVAL,
        mc_n=100,
        std=0.5
    )
    
    # 3. GFM via Distance MC Guidance (Scale = 3.0 on normalized distance, guide_scale = 5.0)
    gfm_distance_trajs[c] = sample_guided_gfm(
        uncond_model, 
        target_class=c, 
        guide_type='distance', 
        scale=8.0, 
        guide_scale=15.0,
        batch_size=BATCH_SIZE_EVAL, 
        num_steps=NUM_STEPS_EVAL,
        mc_n=100,
        std=0.2
    )

## 6. Plotting Generated Samples
Let's display the final scatter plots of the generated points at $t=1.0$ for the three methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 8))

# 1. CCFM
for c in range(8):
    x1_ccfm = ccfm_trajs[c][-1].cpu().numpy()
    axes[0].scatter(x1_ccfm[:, 0], x1_ccfm[:, 1], color=colors[c], alpha=0.5, s=12, label=f'Class {c}' if c==0 else "")
axes[0].set_title('Explicit CCFM Samples', fontsize=14)
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)
axes[0].set_aspect('equal')
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. GFM Indicator
for c in range(8):
    x1_gfm = gfm_indicator_trajs[c][-1].cpu().numpy()
    axes[1].scatter(x1_gfm[:, 0], x1_gfm[:, 1], color=colors[c], alpha=0.5, s=12)
axes[1].set_title('GFM (Indicator MC Guidance) Samples', fontsize=14)
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect('equal')
axes[1].grid(True, linestyle='--', alpha=0.5)

# 3. GFM Distance
for c in range(8):
    x1_gfm_dist = gfm_distance_trajs[c][-1].cpu().numpy()
    axes[2].scatter(x1_gfm_dist[:, 0], x1_gfm_dist[:, 1], color=colors[c], alpha=0.5, s=12)
axes[2].set_title('GFM (Distance MC Guidance) Samples', fontsize=14)
axes[2].set_xlim(-2, 2)
axes[2].set_ylim(-2, 2)
axes[2].set_aspect('equal')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Trajectories Visualization
We plot the flow trajectories of 100 particles moving from $p_0$ to mode index 3.

In [ ]:
target_mode = 3
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot CCFM Trajectories
traj_ccfm = ccfm_trajs[target_mode].cpu().numpy()
for i in range(min(100, BATCH_SIZE_EVAL)):
    axes[0].plot(traj_ccfm[:, i, 0], traj_ccfm[:, i, 1], color=colors[target_mode], alpha=0.2)
axes[0].scatter(traj_ccfm[0, :100, 0], traj_ccfm[0, :100, 1], color='black', s=5, zorder=5, label='Start ($t=0$)')
axes[0].scatter(traj_ccfm[-1, :100, 0], traj_ccfm[-1, :100, 1], color='red', s=12, zorder=5, label='End ($t=1$)')
axes[0].set_title(f'CCFM Trajectories (Mode {target_mode})', fontsize=14)
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)
axes[0].set_aspect('equal')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower left')

# Plot GFM Indicator Trajectories
traj_gfm = gfm_indicator_trajs[target_mode].cpu().numpy()
for i in range(min(100, BATCH_SIZE_EVAL)):
    axes[1].plot(traj_gfm[:, i, 0], traj_gfm[:, i, 1], color=colors[target_mode], alpha=0.2)
axes[1].scatter(traj_gfm[0, :100, 0], traj_gfm[0, :100, 1], color='black', s=5, zorder=5)
axes[1].scatter(traj_gfm[-1, :100, 0], traj_gfm[-1, :100, 1], color='red', s=12, zorder=5)
axes[1].set_title(f'GFM Indicator Trajectories (Mode {target_mode})', fontsize=14)
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect('equal')
axes[1].grid(True, linestyle='--', alpha=0.5)

# Plot GFM Distance Trajectories
traj_gfm_dist = gfm_distance_trajs[target_mode].cpu().numpy()
for i in range(min(100, BATCH_SIZE_EVAL)):
    axes[2].plot(traj_gfm_dist[:, i, 0], traj_gfm_dist[:, i, 1], color=colors[target_mode], alpha=0.2)
axes[2].scatter(traj_gfm_dist[0, :100, 0], traj_gfm_dist[0, :100, 1], color='black', s=5, zorder=5)
axes[2].scatter(traj_gfm_dist[-1, :100, 0], traj_gfm_dist[-1, :100, 1], color='red', s=12, zorder=5)
axes[2].set_title(f'GFM Distance Trajectories (Mode {target_mode})', fontsize=14)
axes[2].set_xlim(-2, 2)
axes[2].set_ylim(-2, 2)
axes[2].set_aspect('equal')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 8. Quantitative Evaluation
We evaluate the quality of the generated samples by computing:
- **Mode Accuracy (%)**: What percentage of generated points end up closest to their target mode center.
- **Cluster spread (Std Dev)**: The standard deviation of generated points from the center of target mode $c$ (Ground Truth = 0.1).
- **Wasserstein-2 Distance**: Distance from the generated distribution to ground-truth samples of target mode $c$.

In [ ]:
def get_nearest_mode(points, centers):
    dists = torch.cdist(points, centers)
    return torch.argmin(dists, dim=1)

centers = x1_dist.centers.to(device)

print(f"{'Method':<20} | {'Mode Acc (%)':<15} | {'Std Dev':<10} | {'W2 Distance':<12}")
print("-" * 65)

methods = [
    ('CCFM', ccfm_trajs),
    ('GFM (Indicator)', gfm_indicator_trajs),
    ('GFM (Distance)', gfm_distance_trajs)
]

for name, trajs in methods:
    accs = []
    stds = []
    w2s = []
    
    for c in range(8):
        x1_samples = trajs[c][-1]
        N = x1_samples.shape[0]
        
        # Mode Accuracy
        nearest = get_nearest_mode(x1_samples, centers)
        acc = (nearest == c).float().mean().item() * 100
        accs.append(acc)
        
        # Mode Spread (Std Dev)
        center_c = centers[c]
        diffs = x1_samples - center_c
        std = torch.std(diffs, dim=0).mean().item()
        stds.append(std)
        
        # W2 Distance
        true_x1 = center_c + torch.randn(N, 2, device=device) * 0.1
        w2_val = compute_w2(x1_samples, true_x1)
        w2s.append(w2_val)

    print(f"{name:<20} | {np.mean(accs):<15.2f} | {np.mean(stds):<10.4f} | {np.mean(w2s):<12.4f}")

## 9. Conclusion & Discussion

1. **Path Straightness**: Explicit Class-Conditional Flow Matching (CCFM) yields extremely straight paths, representing the optimal transport character. Guided Flow Matching (GFM) shows curved paths, as the model unconditionally flows outward first before the Monte Carlo drift correction bends the trajectory towards the target mode.
2. **Mode Accuracy**: CCFM achieves near 100% accuracy automatically since class information is built into the vector field. GFM can achieve high mode accuracy, but is heavily dependent on the chosen guidance scale and standard deviation of candidate endpoints.
3. **Inference Cost**: CCFM is computationally efficient at inference time, running in a single forward pass per ODE step. GFM is significantly slower because it must generate and evaluate $M = 100$ forward simulation endpoints at *every* integration step to calculate the guidance update.